# iLQRの一反復の全体像

ここでは各処理の具体的な導出は行わず、iLQRの一反復で何を行っているのかを理解する。

ここでいう反復とは、実システムにおける制御周期の繰り返しではなく、1つの有限ホライゾン最適制御問題の中で、状態・入力軌道を改善するための最適化反復を意味する。
そして、一反復は「1つの最適軌道を求めるために、**基準軌道**を一度改善する処理」という意味である。この基準軌道はLQRが与えられた目標に到達するように修正していくものであり、反復終了時は最適軌道となる。

----------------------------------------------------------
第 $i$ 反復において、現在の基準軌道

$$
\left(\bar X^{(i)},\bar U^{(i)}\right)
$$

が与えられているとする。iLQRの一反復では、次の処理を行う。

1. 基準軌道上で、非線形動力学モデルを一次近似し、コストを二次近似した局所近似を行う。

2. 局所近似によって得られた線形・二次問題に対してbackward passを行い、以下の入力修正則を構成する $d_k,K_k$ を求める。

   $$
   \delta u_k=d_k+K_k\delta x_k
   $$

3. ラインサーチ係数 $\alpha$ を選び、入力修正則を用いて元の非線形モデルをforward passする。これにより、候補状態軌道と候補入力軌道を求める。

   $$
   X^{\mathrm{cand}}(\alpha),\qquad
   U^{\mathrm{cand}}(\alpha)
   $$

4. 元のコスト関数を使って候補軌道の総コスト

   $$
   J^{\mathrm{cand}}(\alpha)
   $$

   を計算し、現在の基準軌道の総コスト $J^{(i)}$ と比較する。

5. 候補軌道のコストが減少しない場合、

   $$
   J^{\mathrm{cand}}(\alpha)\geq J^{(i)}
   $$

   ならば、$\alpha$ を小さくして手順3のforward passからやり直す。このラインサーチは、あらかじめ定めた回数または最小の $\alpha$ まで繰り返す。

6. コストが減少する候補軌道が得られた場合、

   $$
   J^{\mathrm{cand}}(\alpha)<J^{(i)}
   $$

   ならば、その候補軌道を次の反復の基準軌道として採用する。

   $$
   \bar X^{(i+1)}   =   X^{\mathrm{cand}}(\alpha)
   $$

   $$
   \bar U^{(i+1)}   =   U^{\mathrm{cand}}(\alpha)
   $$

   ここで第 $i$ 反復が終了し、最適化反復を $i+1$ に進める。

コストが減少する候補軌道をラインサーチで見つけられなかった場合は、基準軌道を更新せず、その反復を失敗とする。実用的なiLQRでは正則化を調整してbackward passからやり直すが、その詳細は後のStepで扱う。

一反復の要点は、大きく次の2点である。

1. 基準軌道の周辺で作った局所的な線形・二次問題から入力修正則を求め、その修正則を用いて元の非線形モデルをforward passする点
2. 元のコスト関数を用いて、現在の基準軌道と、$\alpha$を変更しながら生成した候補軌道を比較し、コストが減少する候補だけを次の基準軌道として採用する点


## 添字の区別

iLQRには2種類の計算プロセスがあり、その区別を添字$k,i$で行う。

- $k = 0, \cdots , N$ : 有限ホライゾン内の時刻
- $i = 0, 1, \cdots $ : 上記で示した基準軌道を改善する最適化反復の反復数

例えば、以下は「iLQRの第$i$反復で使っている、有限ホライゾン内の時刻$k$における基準入力」を表す。基準軌道であることは上線(バー)で表す。

$$
\bar{u}_k^{(i)}
$$

入力列全体は$\bar{U}^{(i)}$で表す。

$$
\bar{U}^{(i)} = [\bar{u}_0^{(i)} , \cdots , \bar{u}_{N-1}^{(i)}]
$$


## 0. 最初の基準軌道を作る

非線形状態方程式を離散時間システムとして以下で表す。

$$
x_{k+1} = f(x_k , u_k)
$$

初期入力列 $\bar{U}^{(0)}$ を用意する。例えば、 $\bar{u}_k^{(0)}=0$としても良い。

この $\bar{U}^{(0)}$ を用いて、非線形状態方程式を初期状態 $x_0$ から順方向に計算を行う。初期状態の $x_0$ は予測ホライゾンの計算を始めるときの実システムの状態である。これを $\bar{x}_0^{(0)} = x_0$ としている。

$$
\bar{x}_{k+1}^{(0)} = f(\bar{x}_k^{(0)} , \bar{u}_k^{(0)}), \quad k = 0 , \cdots, N-1
$$

これより、最適化反復 0 回目の基準状態軌道が得られる。

$$
\bar{X}^{(0)} = [\bar{x}_0^{(0)}, \cdots, \bar{x}_N^{(0)}]
$$

この処理がrolloutである。

ここで、現在の状態と入力を合わせた基準軌道が完成する。

$$
\left( \bar{X}^{(0)}, \bar{U}^{(0)}\right)
$$

## 1. 基準軌道の周辺を局所近似する

$(i)$番目の反復として考える。

基準軌道$(\bar{X}^{(i)}, \bar{U}^{(i)})$が与えられているとする。

この基準軌道の周辺で非線形動力学モデルを一次近似し、コストを二次近似する。

### 非線形動力学モデルの基準軌道の周辺で一次近似する

非線形動力学モデルは離散時間状態方程式で次のように表す。

$$
x_{k+1} = f(x_{k} , u_{k})
$$

$(x_{k}, u_{k})$を次のように基準軌道からの近傍点として表し、Taylor展開を行うことで、

$$
x_{k} = \bar{x}_k^{(i)} + \delta x_k \\
u_{k} = \bar{u}_k^{(i)} + \delta u_k
$$

以下の式となる。

$$
x_{k+1} \approx f(\bar{x}_k^{(i)}, \bar{u}_k^{(i)} ) + A_k^{(i)} \delta x_k + B_k^{(i)} \delta u_k
$$

$$
A_k^{(i)} = \left. \frac{\partial f}{\partial x} \right|_{\bar{x}_k^{(i)}, \bar{u}_k^{(i)}} , \quad B_k^{(i)} = \left. \frac{\partial f}{\partial u} \right|_{\bar{x}_k^{(i)}, \bar{u}_k^{(i)}} 
$$


基準軌道では、以下のように非線形動力学モデルの関係があるため、

$$
\bar{x}_{k+1} = f(\bar{x}_k^{(i)}, \bar{u}_k^{(i)} )
$$

以下のように表現される。

$$
x_{k+1} \approx \bar{x}_{k+1} + A_k^{(i)} \delta x_k + B_k^{(i)} \delta u_k
$$

基準軌道の近傍点は次のように$\delta x_k, \delta u_k$ で表すことが出来る。 

$$
\begin{split}
x_{k} = \bar{x}_k^{(i)} + \delta x_k \\
u_{k} = \bar{u}_k^{(i)} + \delta u_k
\end{split} \Rightarrow
\begin{split}
\delta x_k = x_{k} - \bar{x}_k^{(i)}  \\
\delta u_k = u_{k} - \bar{u}_k^{(i)}  
\end{split}
$$

よって、

$$
x_{k+1} -  \bar{x}_{k+1} \approx  A_k^{(i)} \delta x_k + B_k^{(i)} \delta u_k = \delta x_{k+1}
$$

つまり、近傍点は以下の一次近似で表される。

$$
\delta x_{k+1} = A_k^{(i)} \delta x_k + B_k^{(i)} \delta u_k
$$

### コストを基準軌道の周辺で二次近似する

iLQRで扱う総コストを、ステージコスト $\ell_k$ と終端コスト $\phi$ を用いて次のように表す。

$$
J(X,U)=\phi(x_N)+\sum_{k=0}^{N-1}\ell_k(x_k,u_k)
$$

Step 2のLQRでは、$\ell_k$ と $\phi$ を最初から二次関数として与えた。一方、iLQRでは、これらを一般の二回微分可能な関数として扱い、現在の基準軌道

$$
\left(\bar X^{(i)},\bar U^{(i)}\right)
$$

の周辺で二次近似する。

基準軌道からの変化量を、

$$
\delta x_k=x_k-\bar x_k^{(i)},
\qquad
\delta u_k=u_k-\bar u_k^{(i)}
$$

とすると、総コストは次のように近似される。

$$
\begin{aligned}
J(X,U) \approx{}& \bar J^{(i)} + \phi_x^\mathsf{T}\delta x_N + \frac{1}{2} \delta x_N^\mathsf{T} \phi_{xx} \delta x_N \\
&+\sum_{k=0}^{N-1} \left\{ \begin{bmatrix} \ell_{x,k}\\
\ell_{u,k} \end{bmatrix}^{\mathsf T}
\begin{bmatrix}
\delta x_k\\
\delta u_k
\end{bmatrix}
\right.
\\
&\qquad\qquad\left.
+
\frac{1}{2}
\begin{bmatrix}
\delta x_k\\
\delta u_k
\end{bmatrix}^{\mathsf T}
\begin{bmatrix}
\ell_{xx,k} & \ell_{xu,k}\\
\ell_{ux,k} & \ell_{uu,k}
\end{bmatrix}
\begin{bmatrix}
\delta x_k\\
\delta u_k
\end{bmatrix}
\right\},
\end{aligned}
$$

ここで、

$$
\bar J^{(i)}=\phi(\bar x_N^{(i)})+\sum_{k=0}^{N-1}\ell_k(\bar x_k^{(i)},\bar u_k^{(i)})
$$

は、現在の基準軌道の総コストである。また、各勾配とHessianは、それぞれ基準点

$$
(\bar x_k^{(i)},\bar u_k^{(i)})
$$

および終端基準状態 $\bar x_N^{(i)}$ で評価する。

このように、一般のコスト関数を基準軌道からの変化量 $\delta x_k,\delta u_k$ に関する二次式として近似する。動力学の一次近似と組み合わせることで、基準軌道周辺の局所的な線形・二次最適制御問題を構成できる。

各微分量の導出と、これらを用いた $Q$ 関数の構成についてはStep 4で扱う。


## 2. backward passで改善則を構成する

局所近似した線形・二次問題に対して、終端 $k=N$から時刻0へ逆向きに計算する。

その結果、各時刻における入力修正則を構成する。

$$
\delta u_k = d_k + K_k \delta x_k
$$

Step2と同様に、backward pass では具体的な入力列が求まるのではなく、以下の入力修正則を構成する係数列が求まる。

$$
d_0, \cdots, d_{N-1} \\
K_0, \cdots, K_{N-1}
$$

#### $d_k$ の役割

$d_k$ は基準入力自体をどちらに変更すればコストが下がるのかを表す。

例えば基準状態から$\delta x_k = 0$ とずれていなくても、以下の値となる。

$$
\delta u_k = d_k
$$

#### $K_k$の役割

$K_k \delta x_k$ は次のforward pass で生成される新しい状態が基準状態からずれたとき、そのズレに応じて未来のコストが小さくなるように入力修正を調整するフィードバック項である。

詳しくはStep 5 で学ぶ。

## 3. 入力修正則を用いて非線形動力学モデルをforward passし、候補軌道を作る

順方向に計算するforward passでは、初期状態は固定されているので、以下とする。

$$
x_0^{\mathrm{cand}} = \bar{x}_0^{(i)} = x_0
$$

各時刻の候補入力を以下のように構成する。第３項が候補軌道と基準軌道の差をフィードバックする$K_k$である。

$$
u_k^{\mathrm{cand}} =\bar{u}_k^{(i)} + \alpha d_k + K_k(x_k^{\mathrm{cand}} - \bar{x}_k^{(i)})
$$

その入力をもとの非線形モデルへ与え、$x_{k+1}^{\mathrm{cand}}$を求める。

$$
x_{k+1}^{\mathrm{cand}} = f(x_k^{\mathrm{cand}}, u_k^{\mathrm{cand}})
$$

以下のような関係であるため、$k=0$から$N-1$まで繰り返すことで、候補軌道が得られる。

$$
\begin{aligned}
u_0^{\mathrm{cand}}(x_0^{\mathrm{cand}}) &\rightarrow \left[x_1^{\mathrm{cand}} = f(x_0^{\mathrm{cand}}, u_0^{\mathrm{cand}})\right] \rightarrow  \\
u_1^{\mathrm{cand}}(x_1^{\mathrm{cand}}) &\rightarrow \left[x_2^{\mathrm{\mathrm{cand}}} = f(x_1^{\mathrm{cand}}, u_1^{\mathrm{cand}})\right] \rightarrow \\
& \ \vdots
\end{aligned}
$$

$u_k^{\mathrm{cand}}$の計算は係数$\alpha$により調整を行う、そのため候補軌道は以下のように$\alpha$の関数と捉える。

$$
U^{\mathrm{cand}}(\alpha), X^{\mathrm{cand}}(\alpha)
$$

この一連の順方向計算をforward pass と呼ぶ。詳しくはStep 6 で学ぶ。

## 4. 元のコスト関数を使って候補軌道の総コストの計算

### 候補軌道のコストを評価する

forward passによって得られた以下の候補軌道を

$$
\left(
X^{\mathrm{cand}}(\alpha),
U^{\mathrm{cand}}(\alpha)
\right)
$$

局所二次近似する前に定義した元のコスト関数$J(X,U)$へ代入する。

$$
J(X,U)=\phi(x_N)+\sum_{k=0}^{N-1}\ell_k(x_k,u_k)
$$

候補軌道の総コストは、以下となる。

$$
J^{\mathrm{cand}}(\alpha)=\phi\left(x_N^{\mathrm{cand}}(\alpha)\right)+
\sum_{k=0}^{N-1}\ell_k\left(x_k^{\mathrm{cand}}(\alpha),u_k^{\mathrm{cand}}(\alpha)\right)
$$

現在の基準軌道の総コストは、

$$
J^{(i)}=\phi\left(\bar x_N^{(i)}\right)+\sum_{k=0}^{N-1}
\ell_k\left(\bar x_k^{(i)},\bar u_k^{(i)}\right)
$$

である。

候補軌道について、

$$
J^{\mathrm{cand}}(\alpha)<J^{(i)}
$$

が成り立てば、以下のように候補軌道を次の反復の基準軌道として採用する。コストが減少しなければ、$\alpha$を小さくしてforward passをやり直す。これをラインサーチと呼ぶ。

$$
\begin{aligned}
\bar{X}^{(i+1)} = \bar{X}^{\mathrm{cand}}(\alpha) \\
\bar{U}^{(i+1)} = \bar{U}^{\mathrm{cand}}(\alpha) \\
\end{aligned}
$$

ここで第$i$反復が終了し、次の最適化反復$i+1$に進む。


以上